# **Analisis Berpasangan Ketahanan Bahasa Lokal — V5 Batch 80**

---

**Proyek:** KP Agentic AI + LLM  
**Tahap:** Analisis pasca-audit V5, tanpa inferensi ulang  
**Dataset:** MASSIVE-Agents — development  
**Ruang lingkup:** 80 variasi × 4 model; analisis sensitivitas pada 70 variasi non-ditahan

Analisis ini mempertahankan hasil Batch 80 sebagai hasil utama. Keputusan audit berbantuan AI digunakan untuk memeriksa sensitivitas hasil terhadap kandidat yang ditahan, tanpa mengubah ground truth atau prediksi model.

## **1. Konfigurasi dan Sumber Data**

---

Data utama berasal dari hasil perbandingan berpasangan Batch 80. Keputusan kualitas variasi diambil dari audit terpadu V5. Dua sumber tersebut dipertemukan berdasarkan `variant_id`.

In [1]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display


def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents, Path(r"D:\KP-Agentic-AI")]:
        if (candidate / "results" / "ketahanan_bahasa_lokal" / "eksplorasi_v5_batch80").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek KP-Agentic-AI tidak ditemukan.")

ROOT = find_project_root()
BATCH_DIR = ROOT / "results" / "ketahanan_bahasa_lokal" / "eksplorasi_v5_batch80"
PAIRED_FILE = BATCH_DIR / "paired_comparison_v5.csv"
SELECTED_FILE = BATCH_DIR / "selected_variants_v5.csv"
AUDIT_FILE = BATCH_DIR / "KP_FINAL_AUDIT_TERPADU_V5_BATCH80.xlsx"
OUTPUT_DIR = BATCH_DIR / "analisis_pasca_audit_v5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

paired_raw = pd.read_csv(PAIRED_FILE, dtype={"variant_id": "string"}).fillna("")
selected_raw = pd.read_csv(SELECTED_FILE, dtype={"variant_id": "string"}).fillna("")
audit_raw = pd.read_excel(AUDIT_FILE, sheet_name="AUDIT_80", header=3).fillna("")

print("Perbandingan prediksi:", len(paired_raw))
print("Kandidat terpilih:", len(selected_raw))
print("Baris audit:", len(audit_raw))

Perbandingan prediksi: 320
Kandidat terpilih: 80
Baris audit: 80


## **2. Pemeriksaan Integritas dan Identitas Pasangan**

---

Setiap pasangan model–variasi harus unik. Seluruh data yang digunakan dalam analisis merupakan bagian dari 80 variasi pada development set, masing-masing dengan empat prediksi model.

In [2]:
assert len(paired_raw) == 320, "Jumlah prediksi tidak sesuai desain Batch 80."
assert len(selected_raw) == 80, "Jumlah variasi terpilih tidak sesuai."
assert len(audit_raw) == 80, "Jumlah keputusan audit tidak sesuai."
assert audit_raw["variant_id"].is_unique
assert selected_raw["variant_id"].is_unique
assert not paired_raw.duplicated(subset=["model", "variant_id"]).any()
assert paired_raw["variant_id"].nunique() == 80
assert set(paired_raw["variant_id"]) == set(selected_raw["variant_id"]) == set(audit_raw["variant_id"])
assert paired_raw.groupby("variant_id")["model"].nunique().eq(4).all()
assert selected_raw["group_id"].nunique() == 80
assert selected_raw["split"].eq("development").all()

for col in ("fsa_original_correct", "fsa_variasi_correct", "ast_original_correct", "ast_variasi_correct"):
    normalized = paired_raw[col].astype(str).str.lower()
    assert normalized.isin(["true", "false"]).all(), f"Status tidak valid: {col}"
    paired_raw[col] = normalized.eq("true")

print("Integritas prediksi, varian, group_id, dan split: PASS")

Integritas prediksi, varian, group_id, dan split: PASS


## **3. Keputusan Audit dan Pembentukan Subset**

---

Keputusan `LOLOS_TINJAUAN_AI` dan `LOLOS_DENGAN_CATATAN` digabungkan sebagai kelompok **non-ditahan** untuk analisis sensitivitas. Status ini bukan validasi manusia dan bukan persetujuan evaluasi final.

In [3]:
audit_cols = ["variant_id", "Kategori", "Status tinjauan AI", "Alasan keputusan berbasis sumber", "Validasi manusia"]
audit = audit_raw[audit_cols].copy()
audit.columns = ["variant_id", "kategori_audit", "status_audit_ai", "alasan_audit", "validasi_manusia"]
audit["kelompok_audit"] = audit["status_audit_ai"].map(
    lambda s: "NON_DITAHAN_AI" if s in ("LOLOS_TINJAUAN_AI", "LOLOS_DENGAN_CATATAN") else "DITAHAN_AI"
)

paired = paired_raw.drop(columns=["validasi_manusia"], errors="ignore").merge(audit, on="variant_id", validate="many_to_one")
selected = selected_raw.drop(columns=["validasi_manusia"], errors="ignore").merge(audit, on="variant_id", validate="one_to_one")
assert len(paired) == 320 and len(selected) == 80
assert set(audit["kelompok_audit"]) == {"NON_DITAHAN_AI", "DITAHAN_AI"}
assert (selected["kelompok_audit"] == "NON_DITAHAN_AI").sum() == 70
assert (selected["kelompok_audit"] == "DITAHAN_AI").sum() == 10
assert paired["validasi_manusia"].astype(str).str.upper().isin(["BELUM", "PENDING", "", "TIDAK"]).all()

ringkasan_audit = (selected.groupby(["kelompok_audit", "kategori", "status_audit_ai"])
                  .size().reset_index(name="n_variasi").sort_values(["kelompok_audit", "kategori", "status_audit_ai"]))
display(ringkasan_audit)
print("Subset sensitivitas: 70 variasi × 4 model = 280 prediksi")

,kelompok_audit,kategori,status_audit_ai,n_variasi
0,DITAHAN_AI,code_switch_id_en,TAHAN_GT_ARGUMEN,1
1,DITAHAN_AI,code_switch_id_en,TAHAN_GT_EN_ID,1
2,DITAHAN_AI,code_switch_id_en,TAHAN_GT_INTENT,2
3,DITAHAN_AI,code_switch_id_en,TAHAN_KONTEKS,1
4,DITAHAN_AI,singkatan,TAHAN_GT_KONTEKS,1
5,DITAHAN_AI,slang,TAHAN_GT_ARGUMEN,1
6,DITAHAN_AI,slang,TAHAN_GT_INTENT,1
7,DITAHAN_AI,typo,TAHAN_GT_ARGUMEN,1
8,DITAHAN_AI,typo,TAHAN_KEWAJARAN_SUMBER,1
9,NON_DITAHAN_AI,code_switch_id_en,LOLOS_DENGAN_CATATAN,2


Subset sensitivitas: 70 variasi × 4 model = 280 prediksi


## **4. Perbandingan AST dan FSA pada Dua Kelompok**

---

Skor instruksi asli dan variasi dihitung pada pasangan yang sama. Hasil lengkap 80 variasi tetap menjadi acuan; hasil 70 variasi merupakan analisis sensitivitas setelah pemeriksaan kualitas data.

In [4]:
def summarize(subset, label):
    n = len(subset)
    original_ast = int(subset["ast_original_correct"].sum())
    variant_ast = int(subset["ast_variasi_correct"].sum())
    original_fsa = int(subset["fsa_original_correct"].sum())
    variant_fsa = int(subset["fsa_variasi_correct"].sum())
    return {"kelompok": label, "n_variasi": subset["variant_id"].nunique(), "n_prediksi": n,
            "ast_asli_benar": original_ast, "ast_variasi_benar": variant_ast,
            "ast_asli_pct": round(original_ast/n*100, 3), "ast_variasi_pct": round(variant_ast/n*100, 3),
            "delta_ast_pp": round((variant_ast-original_ast)/n*100, 3),
            "fsa_asli_benar": original_fsa, "fsa_variasi_benar": variant_fsa,
            "fsa_asli_pct": round(original_fsa/n*100, 3), "fsa_variasi_pct": round(variant_fsa/n*100, 3),
            "delta_fsa_pp": round((variant_fsa-original_fsa)/n*100, 3)}

all80 = paired.copy()
non_hold70 = paired.loc[paired["kelompok_audit"] == "NON_DITAHAN_AI"].copy()
held10 = paired.loc[paired["kelompok_audit"] == "DITAHAN_AI"].copy()
summary_groups = pd.DataFrame([
    summarize(all80, "BATCH80_UTUH"),
    summarize(non_hold70, "NON_DITAHAN_AI_70"),
    summarize(held10, "DITAHAN_AI_10")
])
assert summary_groups.loc[0, "ast_asli_benar"] == 45
assert summary_groups.loc[0, "ast_variasi_benar"] == 39
assert summary_groups.loc[1, "ast_asli_benar"] == 44
assert summary_groups.loc[1, "ast_variasi_benar"] == 37
assert summary_groups.loc[0, "fsa_asli_benar"] == 230
assert summary_groups.loc[0, "fsa_variasi_benar"] == 212
display(summary_groups)

,kelompok,n_variasi,n_prediksi,ast_asli_benar,ast_variasi_benar,ast_asli_pct,ast_variasi_pct,delta_ast_pp,fsa_asli_benar,fsa_variasi_benar,fsa_asli_pct,fsa_variasi_pct,delta_fsa_pp
0,BATCH80_UTUH,80,320,45,39,14.062,12.188,-1.875,230,212,71.875,66.250,-5.625
1,NON_DITAHAN_AI_70,70,280,44,37,15.714,13.214,-2.500,214,200,76.429,71.429,-5.000
2,DITAHAN_AI_10,10,40,1,2,2.500,5.000,2.500,16,12,40.000,30.000,-10.000


## **5. Hasil per Model dan Kategori**

---

Evaluasi dikelompokkan per model dan kategori bahasa. Jumlah variasi per kategori pada subset 70 dapat berbeda karena penahanan kandidat berdasarkan kualitas data, sehingga perubahan persentase tidak dibandingkan secara langsung tanpa melihat penyebutnya.

In [5]:
rows = []
for label, group in [("BATCH80_UTUH", all80), ("NON_DITAHAN_AI_70", non_hold70)]:
    for model, sub in group.groupby("model", sort=False):
        rows.append({"dimensi": "model", "nama": model, **summarize(sub, label)})
    for category, sub in group.groupby("kategori", sort=False):
        rows.append({"dimensi": "kategori", "nama": category, **summarize(sub, label)})
    for (model, category), sub in group.groupby(["model", "kategori"], sort=False):
        rows.append({"dimensi": "model_kategori", "nama": f"{model} | {category}", **summarize(sub, label)})
summary_breakdown = pd.DataFrame(rows)
display(summary_breakdown.loc[summary_breakdown["dimensi"] == "model",
    ["kelompok", "nama", "n_variasi", "ast_asli_benar", "ast_variasi_benar", "delta_ast_pp", "delta_fsa_pp"]])
display(summary_breakdown.loc[summary_breakdown["dimensi"] == "kategori",
    ["kelompok", "nama", "n_variasi", "ast_asli_benar", "ast_variasi_benar", "delta_ast_pp", "delta_fsa_pp"]])

,kelompok,nama,n_variasi,ast_asli_benar,ast_variasi_benar,delta_ast_pp,delta_fsa_pp
0,BATCH80_UTUH,qwen2.5:7b-instruct,80,17,15,-2.500,-8.750
1,BATCH80_UTUH,gemma2:9b-instruct-q4_K_M,80,8,8,0.000,-3.750
2,BATCH80_UTUH,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,80,15,15,0.000,0.000
3,BATCH80_UTUH,lfm2.5:8b-a1b-q4_K_M,80,5,1,-5.000,-10.000
24,NON_DITAHAN_AI_70,qwen2.5:7b-instruct,70,17,15,-2.857,-7.143
25,NON_DITAHAN_AI_70,gemma2:9b-instruct-q4_K_M,70,8,8,0.000,-1.429
26,NON_DITAHAN_AI_70,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,70,14,13,-1.429,1.429
27,NON_DITAHAN_AI_70,lfm2.5:8b-a1b-q4_K_M,70,5,1,-5.714,-12.857


,kelompok,nama,n_variasi,ast_asli_benar,ast_variasi_benar,delta_ast_pp,delta_fsa_pp
4,BATCH80_UTUH,slang,20,20,18,-2.500,-7.500
5,BATCH80_UTUH,singkatan,20,5,5,0.000,0.000
6,BATCH80_UTUH,typo,20,12,7,-6.250,-17.500
7,BATCH80_UTUH,code_switch_id_en,20,8,9,1.250,2.500
28,NON_DITAHAN_AI_70,slang,18,20,18,-2.778,-8.333
29,NON_DITAHAN_AI_70,singkatan,19,5,4,-1.316,0.000
30,NON_DITAHAN_AI_70,typo,18,12,7,-6.944,-18.056
31,NON_DITAHAN_AI_70,code_switch_id_en,15,7,8,1.667,8.333


## **6. Transisi Prediksi Berpasangan**

---

Perubahan status dipisahkan menjadi benar→salah, salah→benar, tetap benar, dan tetap salah. Pemeriksaan dilakukan untuk AST dan FSA agar perubahan argumen tidak disamakan dengan perubahan pemilihan fungsi.

In [6]:
def transition(a, b):
    if a and b: return "tetap_benar"
    if a and not b: return "benar_ke_salah"
    if not a and b: return "salah_ke_benar"
    return "tetap_salah"

for metric in ["ast", "fsa"]:
    paired[f"transisi_{metric}"] = [transition(a, b) for a, b in zip(
        paired[f"{metric}_original_correct"], paired[f"{metric}_variasi_correct"])]
transisi_rows = []
for label, sub in [("BATCH80_UTUH", paired),
                   ("NON_DITAHAN_AI_70", paired[paired["kelompok_audit"] == "NON_DITAHAN_AI"]),
                   ("DITAHAN_AI_10", paired[paired["kelompok_audit"] == "DITAHAN_AI"])]:
    for metric in ["ast", "fsa"]:
        ct = sub[f"transisi_{metric}"].value_counts().to_dict()
        transisi_rows.append({"kelompok": label, "metrik": metric, "n": len(sub),
            **{x: ct.get(x, 0) for x in ["tetap_benar", "benar_ke_salah", "salah_ke_benar", "tetap_salah"]}})
transisi_summary = pd.DataFrame(transisi_rows)
assert transisi_summary.loc[0, "benar_ke_salah"] == 14
assert transisi_summary.loc[0, "salah_ke_benar"] == 8
display(transisi_summary)

,kelompok,metrik,n,tetap_benar,benar_ke_salah,salah_ke_benar,tetap_salah
0,BATCH80_UTUH,ast,320,31,14,8,267
1,BATCH80_UTUH,fsa,320,202,28,10,80
2,NON_DITAHAN_AI_70,ast,280,31,13,6,230
3,NON_DITAHAN_AI_70,fsa,280,191,23,9,57
4,DITAHAN_AI_10,ast,40,0,1,2,37
5,DITAHAN_AI_10,fsa,40,11,5,1,23


## **7. Penelusuran Kasus Perubahan**

---

Kasus dengan perubahan status atau fungsi dipisahkan dari kasus yang nilainya tetap. Informasi fungsi target, fungsi prediksi, galat AST, dan status audit digunakan untuk menelusuri penyebab perubahan. Perubahan prediksi tidak otomatis membuktikan kerusakan semantik instruksi.

In [7]:
paired["pergantian_fungsi"] = paired["fungsi_original"].astype(str).ne(paired["fungsi_variasi"].astype(str))
prioritas_perubahan = paired.loc[
    (paired["transisi_ast"].isin(["benar_ke_salah", "salah_ke_benar"])) |
    (paired["pergantian_fungsi"]),
    ["variant_id", "model", "kategori", "kelompok_audit", "status_audit_ai", "alasan_audit",
     "fungsi_gt", "fungsi_original", "fungsi_variasi", "pergantian_fungsi",
     "transisi_ast", "transisi_fsa", "ast_error_type", "ast_error_message"]
].copy()
held_variants = selected.loc[selected["kelompok_audit"] == "DITAHAN_AI",
    ["variant_id", "kategori", "instruksi_id_asli", "instruksi_variasi", "fungsi_gt",
     "status_audit_ai", "alasan_audit", "validasi_manusia"]].copy()
print("Prediksi dengan perubahan AST atau fungsi:", len(prioritas_perubahan))
print("Variasi unik dengan perubahan:", prioritas_perubahan["variant_id"].nunique())
print("Variasi ditahan dari subset sensitivitas:", len(held_variants))
display(held_variants[["variant_id", "kategori", "status_audit_ai", "alasan_audit"]])

Prediksi dengan perubahan AST atau fungsi: 76
Variasi unik dengan perubahan: 45
Variasi ditahan dari subset sensitivitas: 10


,variant_id,kategori,status_audit_ai,alasan_audit
0,id-ID:0001210::slang,slang,TAHAN_GT_ARGUMEN,"GT calendar.query memberi event_name='temui', ..."
15,id-ID:0000309::slang,slang,TAHAN_GT_INTENT,ID 'masukkan ke daftar putar cinta' dapat bera...
35,id-ID:0001256::singkatan,singkatan,TAHAN_GT_KONTEKS,'Memulai polling untuk email baru' dapat berar...
49,id-ID:0000626::typo,typo,TAHAN_GT_ARGUMEN,Instruksi meminta pengingat satu jam sebelum r...
51,id-ID:0000126::typo,typo,TAHAN_KEWAJARAN_SUMBER,Sumber sudah mengandung kesalahan 'cerahakan';...
65,id-ID:0000201::code_switch_id_en,code_switch_id_en,TAHAN_GT_INTENT,'Tampilkan warna redup padaku' dan EN 'show me...
68,id-ID:0000918::code_switch_id_en,code_switch_id_en,TAHAN_GT_EN_ID,EN 'start the gilmore guys' berbeda dari ID 'm...
71,id-ID:0000857::code_switch_id_en,code_switch_id_en,TAHAN_KONTEKS,"EN 'play racing' mendukung permainan, tetapi I..."
74,id-ID:0000756::code_switch_id_en,code_switch_id_en,TAHAN_GT_ARGUMEN,Instruksi ID menyebut stasiun 'radio kosmonita...
79,id-ID:0001224::code_switch_id_en,code_switch_id_en,TAHAN_GT_INTENT,ID meminta membuka kontak dan menelepon Joni B...


## **8. Pemeriksaan Konsistensi Hasil dan Batas Interpretasi**

---

Analisis utuh dan analisis sensitivitas memiliki populasi berbeda. Subset 70 dipilih setelah audit berbantuan AI sehingga tidak menggantikan hasil utama 80 variasi; keduanya belum dapat digunakan untuk menyimpulkan dampak bahasa secara kausal atau mengklaim signifikansi statistik.

In [8]:
assert len(all80) == 320 and len(non_hold70) == 280 and len(held10) == 40
assert len(set(non_hold70["variant_id"])) == 70
assert len(set(held10["variant_id"])) == 10
assert set(non_hold70["variant_id"]).isdisjoint(set(held10["variant_id"]))
assert len(set(non_hold70["group_id"])) == 70
assert len(set(held10["group_id"])) == 10

# Semua analisis hanya membaca hasil prediksi yang sudah dibuat pada eksperimen sebelumnya.
print("Audit + paired output: sesuai (80/70/10 variasi)")
print("Inferensi ulang: tidak dilakukan")
print("Ground truth sumber: tidak diubah")
print("Validasi manusia: belum dilakukan")

Audit + paired output: sesuai (80/70/10 variasi)
Inferensi ulang: tidak dilakukan
Ground truth sumber: tidak diubah
Validasi manusia: belum dilakukan


## **9. Penyimpanan Hasil Analisis**

---

Hasil dibukukan dalam ringkasan gabungan, rincian model dan kategori, transisi status, daftar bukti prediksi yang berubah, serta data yang ditahan. File sumber Batch 80 tetap dipertahankan.

In [9]:
output_tables = {
    "ringkasan_80_70_10.csv": summary_groups,
    "hasil_per_model_dan_kategori.csv": summary_breakdown,
    "transisi_berpasangan.csv": transisi_summary,
    "bukti_perubahan_prediksi.csv": prioritas_perubahan,
    "variasi_ditahan_10.csv": held_variants,
    "hasil_berpasangan_dengan_audit.csv": paired,
}
for name, frame in output_tables.items():
    frame.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8-sig")
print("Jumlah tabel analisis tersimpan:", len(output_tables))
print("Lokasi:", OUTPUT_DIR)

Jumlah tabel analisis tersimpan: 6
Lokasi: D:\KP-Agentic-AI\results\ketahanan_bahasa_lokal\eksplorasi_v5_batch80\analisis_pasca_audit_v5


## **10. Ringkasan Penelitian dan Metadata**

---

Analisis ini menghitung ulang hasil pengujian yang sudah tersedia tanpa membuat prediksi baru. Pelaporan utama tetap menggunakan Batch 80, sedangkan subset 70 menunjukkan sensitivitas terhadap keputusan audit. Interpretasi final menunggu adjudikasi dan validasi manusia sesuai protokol penelitian.

In [10]:
metadata = {
    "run": "analisis_pasca_audit_v5_batch80",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_split": "development",
    "batch_variants": 80,
    "batch_predictions": 320,
    "not_held_ai_variants": 70,
    "not_held_ai_predictions": 280,
    "held_ai_variants": 10,
    "held_ai_predictions": 40,
    "source_files_sha256": {
        "paired_comparison_v5.csv": hashlib.sha256(PAIRED_FILE.read_bytes()).hexdigest(),
        "selected_variants_v5.csv": hashlib.sha256(SELECTED_FILE.read_bytes()).hexdigest(),
        "KP_FINAL_AUDIT_TERPADU_V5_BATCH80.xlsx": hashlib.sha256(AUDIT_FILE.read_bytes()).hexdigest(),
    },
    "model_inference_repeated": False,
    "gt_modified": False,
    "test_used": False,
    "human_validation": "PENDING",
    "status": "eksploratif_analisis_sensitivitas_pasca_audit",
    "note": "Subset 70 non-ditahan oleh tinjauan AI bukan dataset berlabel validasi manusia."
}
(OUTPUT_DIR / "metadata_analisis.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("AST — batch utuh:", f"{summary_groups.loc[0, 'ast_asli_pct']:.2f}% → {summary_groups.loc[0, 'ast_variasi_pct']:.2f}%")
print("AST — subset 70:", f"{summary_groups.loc[1, 'ast_asli_pct']:.2f}% → {summary_groups.loc[1, 'ast_variasi_pct']:.2f}%")
print("FSA — batch utuh:", f"{summary_groups.loc[0, 'fsa_asli_pct']:.2f}% → {summary_groups.loc[0, 'fsa_variasi_pct']:.2f}%")
print("FSA — subset 70:", f"{summary_groups.loc[1, 'fsa_asli_pct']:.2f}% → {summary_groups.loc[1, 'fsa_variasi_pct']:.2f}%")

AST — batch utuh: 14.06% → 12.19%
AST — subset 70: 15.71% → 13.21%
FSA — batch utuh: 71.88% → 66.25%
FSA — subset 70: 76.43% → 71.43%
